# ID Forensics — Colab Workbench

Run only the sections you need. Each section is independent.

**Persistent on Drive** (`My Drive/id-forensics/`): images, weights, eval reports

**Colab Secrets** (🔑 sidebar): `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION`, `AWS_SESSION_TOKEN`

## Config

In [ ]:
GITHUB_TOKEN = ""   # only if repo is private

# --- Data pipeline toggles ---
SYNC_IMAGES = False     # True: download new images S3→Drive
REBUILD_DATASET = True  # True: convert labels → YOLO splits

# --- Training — list stages to run; others are skipped ---
# Options: "stage1", "stage2", "stage4"
# stage3 (tampering) is algorithmic only — no ML training yet
TRAIN_STAGES = ["stage2"]

RUN_EVAL = True
EVAL_SPLIT = "val"

# Hyperparameters
DEVICE = "cuda"
STAGE1_EPOCHS, STAGE1_BATCH, STAGE1_MODEL = 100, 16, "yolov8s-pose.pt"
STAGE2_EPOCHS, STAGE2_BATCH = 40, 32
STAGE4_EPOCHS, STAGE4_BATCH = 50, 32

---
## 1. Connect — Drive + GitHub + deps

Mounts Drive, pulls latest code + labels from GitHub, installs Python packages.
Run every session.

In [ ]:
import os, sys, importlib, subprocess
from pathlib import Path

import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

REPO = Path("/content/id-forensics-model")
user, repo = "ansisvaisla", "id-forensics-model"
url = (
    f"https://{GITHUB_TOKEN}@github.com/{user}/{repo}.git"
    if GITHUB_TOKEN else f"https://github.com/{user}/{repo}.git"
)

os.chdir("/content")
if not REPO.is_dir():
    subprocess.run(["git", "clone", url, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "origin", "main"], check=False)

sys.path.insert(0, str(REPO / "scripts"))
import colab_bootstrap as cb
importlib.reload(cb)

cb.connect(github_token=GITHUB_TOKEN)

---
## 2. Sync images from S3 → Drive

**When to run:** first time, or after adding new labeled images.

In [ ]:
if SYNC_IMAGES:
    cb.sync_images_from_s3()
else:
    n = len(list(cb.DRIVE_RAW_DIR.glob("*.jpg")))
    print(f"Skipped S3 sync — {n} images already on Drive")

---
## 3. Build training datasets

**When to run:** after pushing new labels to GitHub, or after syncing new images.

In [ ]:
if REBUILD_DATASET:
    cb.rebuild_splits()
else:
    print("Skipped dataset rebuild")

---
## 4. Train models

Only stages listed in `TRAIN_STAGES` run. Others print "Skipping".

### Stage 1 — corner detection (YOLO Pose)

In [ ]:
if "stage1" in TRAIN_STAGES:
    cb.clear_corner_caches()
    !cd /content/id-forensics-model && python scripts/training/train_stage1_corners.py \
        --model {STAGE1_MODEL} --device 0 --epochs {STAGE1_EPOCHS} --batch {STAGE1_BATCH}
else:
    print("Skipping Stage 1")

### Stage 2 — presentation attack (screen / printout / live)

In [ ]:
if "stage2" in TRAIN_STAGES:
    !cd /content/id-forensics-model && python scripts/training/train_stage2_screen.py \
        --device {DEVICE} --epochs {STAGE2_EPOCHS} --batch {STAGE2_BATCH}
else:
    print("Skipping Stage 2")

### Stage 3 — tampering / injection (ELA + EXIF)

No ML model to train — algorithmic only. Runs in the pipeline automatically.

In [ ]:
if "stage3" in TRAIN_STAGES:
    print("Stage 3 has no training script — tampering detection is rule-based (ELA + EXIF).")
    print("Remove 'stage3' from TRAIN_STAGES.")
else:
    print("Skipping Stage 3 (no ML model)")

### Stage 4 — ID type classifier (EfficientNet-B0)

Requires Stage 1 weights for deskewing. Load from Drive if not trained this session.

In [ ]:
if "stage4" in TRAIN_STAGES:
    from pathlib import Path
    import shutil

    # Copy Stage 1 weights from Drive if missing locally
    s1_local = Path("/content/id-forensics-model/models/stage1_corners/weights/best.pt")
    s1_drive = cb.OUTPUTS_DIR / "stage1_best.pt"
    if not s1_local.is_file() and s1_drive.is_file():
        s1_local.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(s1_drive, s1_local)
        print(f"Loaded Stage 1 weights from Drive: {s1_drive}")
    elif not s1_local.is_file():
        print("WARNING: Stage 1 weights missing — deskew may fail. Train stage1 first or save weights to Drive.")

    !cd /content/id-forensics-model && python scripts/deskew_id_type_images.py
    !cd /content/id-forensics-model && python scripts/split_id_type_dataset.py --source all_deskewed
    !cd /content/id-forensics-model && python scripts/training/train_stage4_id_type.py \
        --device {DEVICE} --epochs {STAGE4_EPOCHS} --batch {STAGE4_BATCH}
else:
    print("Skipping Stage 4")

---
## 5. Save weights to Drive

In [ ]:
for stage in TRAIN_STAGES:
    if stage == "stage3":
        continue
    try:
        cb.save_weights(stage)
    except FileNotFoundError as exc:
        print(f"Could not save {stage}: {exc}")

---
## 6. Evaluate (saved to Drive)

In [ ]:
if RUN_EVAL:
    for stage in TRAIN_STAGES:
        if stage == "stage3":
            print("Skipping Stage 3 eval (no ML model)")
            continue
        try:
            eval_dir = cb.run_eval(stage, split=EVAL_SPLIT)
            print(f"  wrong images: {eval_dir / 'viz' / 'wrong'}")
        except Exception as exc:
            print(f"Eval failed for {stage}: {exc}")
else:
    print("Skipping eval")